In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#### Constants

In [ ]:
str_bucket = 'assessment-alt'
str_task = '03_data_split'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

#### Load Data

In [ ]:
%%time
df = pd.read_csv(f's3://{str_bucket}/00_data_collection/transaction_data.csv', index_col=0)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')

#### Date Distribution

In [ ]:
sr_monthly = df.set_index('date').resample('M').size()

fig, ax = plt.subplots(figsize=(14, 5))
sr_monthly.plot(ax=ax, color='steelblue', linewidth=2)
ax.set_title('Transactions Over Time (Monthly)')
ax.set_xlabel('Date')
ax.set_ylabel('Transaction Count')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/date_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

#### Define Split Points

Using out-of-time split to prevent lookahead bias:
- **Train**: before 2021-06-01 (~70%)
- **Valid**: 2021-06-01 to 2021-11-01 (~15%)
- **Test**: 2021-11-01 onwards (~15%)

In [ ]:
dt_valid_start = pd.Timestamp('2021-06-01')
dt_test_start = pd.Timestamp('2021-11-01')

df_train = df[df['date'] < dt_valid_start].copy()
df_valid = df[(df['date'] >= dt_valid_start) & (df['date'] < dt_test_start)].copy()
df_test = df[df['date'] >= dt_test_start].copy()

print(f'Train: {df_train.shape[0]:,} rows ({df_train.shape[0]/len(df)*100:.1f}%)')
print(f'Valid: {df_valid.shape[0]:,} rows ({df_valid.shape[0]/len(df)*100:.1f}%)')
print(f'Test:  {df_test.shape[0]:,} rows ({df_test.shape[0]/len(df)*100:.1f}%)')
print(f'\nTrain: {df_train["date"].min()} to {df_train["date"].max()}')
print(f'Valid: {df_valid["date"].min()} to {df_valid["date"].max()}')
print(f'Test:  {df_test["date"].min()} to {df_test["date"].max()}')

#### Verify Distributions

In [ ]:
# Price distribution across splits
print('Price Distribution:')
df_price_dist = pd.DataFrame({
    'Train': df_train['price'].describe(),
    'Valid': df_valid['price'].describe(),
    'Test': df_test['price'].describe(),
})
print(df_price_dist.round(2).to_string())

In [ ]:
# Grade distribution across splits
print('Grade Distribution (%):')
df_grade_dist = pd.DataFrame({
    'Train': df_train['grade'].value_counts(normalize=True).sort_index(),
    'Valid': df_valid['grade'].value_counts(normalize=True).sort_index(),
    'Test': df_test['grade'].value_counts(normalize=True).sort_index(),
})
print((df_grade_dist * 100).round(1).to_string())

In [ ]:
# Grading company distribution across splits
print('Grading Company Distribution (%):')
df_gc_dist = pd.DataFrame({
    'Train': df_train['grading_company'].value_counts(normalize=True),
    'Valid': df_valid['grading_company'].value_counts(normalize=True),
    'Test': df_test['grading_company'].value_counts(normalize=True),
})
print((df_gc_dist * 100).round(1).to_string())

#### Visualize Splits

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for str_label, df_split, str_color in [('Train', df_train, 'steelblue'),
                                         ('Valid', df_valid, 'orange'),
                                         ('Test', df_test, 'green')]:
    sr_monthly = df_split.set_index('date').resample('M').size()
    sr_monthly.plot(ax=ax, color=str_color, linewidth=2, label=str_label)

ax.axvline(dt_valid_start, color='orange', linestyle='--', alpha=0.7, label='Valid Start')
ax.axvline(dt_test_start, color='green', linestyle='--', alpha=0.7, label='Test Start')
ax.set_title('Transaction Volume by Split')
ax.set_xlabel('Date')
ax.set_ylabel('Transaction Count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

#### Save to S3

In [ ]:
%%time
for str_name, df_split in [('train', df_train), ('valid', df_valid), ('test', df_test)]:
    str_s3_path = f's3://{str_bucket}/{str_task}/{str_name}.csv'
    df_split.to_csv(str_s3_path, index=False)
    print(f'Saved {str_name}.csv to {str_s3_path} ({df_split.shape})')

#### Takeaways

- **Out-of-time split** used to prevent lookahead bias — critical for time-series pricing data
- **Train/Valid/Test** roughly 70/15/15 split by date
- **Price distributions** are similar across splits, though median prices may shift over time due to market trends
- **Grade and grading company distributions** are relatively stable across splits
- Temporal split means the model must generalize to future market conditions, not just unseen random samples